# Phase 3 — LSTM Model in PyTorch
**Goal:** Build, train and evaluate an LSTM that predicts whether TSLA's price will go UP or DOWN the next day.

**Why LSTM?**
Stock prices are sequential — what happened yesterday affects today. LSTMs are designed specifically for this: they have a memory cell that learns which past information to keep and which to forget. A regular neural network treats each day independently; LSTM understands the sequence.

**What we predict:** Price direction (UP=1 / DOWN=0) — this is a binary classification problem. Easier to evaluate than exact price regression and more useful for trading decisions.

## Step 1 — Install and import

In [ ]:
!pip install torch pandas scikit-learn matplotlib numpy --quiet
print('Done!')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
# This ensures you get the same results every run
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Step 2 — Load and inspect the combined dataset

In [ ]:
os.chdir('/content')
df = pd.read_csv('tsla_combined.csv')

print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head(3)

## Step 3 — Create the target variable

We predict whether tomorrow's close price is HIGHER than today's.
- 1 = price went UP tomorrow
- 0 = price went DOWN tomorrow

In [ ]:
# Shift Close by -1 to get tomorrow's price
df['next_close'] = df['Close'].shift(-1)

# 1 if tomorrow's price > today's price, else 0
df['target'] = (df['next_close'] > df['Close']).astype(int)

# Drop the last row (no tomorrow for the final day)
df.dropna(subset=['next_close'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Dataset size after target creation: {len(df)}')
print(f'\nTarget distribution:')
print(f'  UP   (1): {df["target"].sum()} days ({df["target"].mean()*100:.1f}%)')
print(f'  DOWN (0): {(1-df["target"]).sum()} days ({(1-df["target"].mean())*100:.1f}%)')
print('\nNote: ~50/50 split is expected for stock direction')

## Step 4 — Select features and scale

We use all price + sentiment features EXCEPT raw price columns (Open/High/Low/Close/Volume).
Raw prices are non-stationary (they trend upward over time) which confuses LSTMs.
We use returns and indicators instead — these are stationary.

In [ ]:
FEATURE_COLS = [
    # Price-derived features (stationary)
    'daily_return', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_mid', 'BB_lower', 'volume_change', 'MA_20', 'MA_50',
    # Sentiment features from FinBERT
    'positive', 'negative', 'neutral', 'compound', 'n_headlines'
]

TARGET_COL = 'target'

print(f'Number of features: {len(FEATURE_COLS)}')
print(f'Features: {FEATURE_COLS}')

# Check for any remaining NaN in features
print(f'\nNaN in features: {df[FEATURE_COLS].isnull().sum().sum()}')

In [ ]:
# Scale features to [0, 1] range
# LSTMs train much better when all features are on the same scale
# IMPORTANT: fit scaler ONLY on training data to avoid data leakage

# First do chronological train/val/test split
# NEVER use random split for time series — future data would leak into training!
n = len(df)
train_end = int(n * 0.70)   # 70% train
val_end   = int(n * 0.85)   # 15% val
# remaining 15% = test

train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

print(f'Train: {len(train_df)} days ({train_df["Date"].min()} to {train_df["Date"].max()})')
print(f'Val:   {len(val_df)} days ({val_df["Date"].min()} to {val_df["Date"].max()})')
print(f'Test:  {len(test_df)} days ({test_df["Date"].min()} to {test_df["Date"].max()})')

# Fit scaler on TRAIN only, then transform all splits
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df[FEATURE_COLS])
val_scaled   = scaler.transform(val_df[FEATURE_COLS])
test_scaled  = scaler.transform(test_df[FEATURE_COLS])

print(f'\nScaling done. Feature range: [{train_scaled.min():.2f}, {train_scaled.max():.2f}]')

## Step 5 — Build sliding window sequences

LSTMs take sequences as input, not individual days.
We use a 30-day lookback window: to predict day T+1, we feed the model days T-29 to T.

```
Input:  [day1, day2, ..., day30]  → shape (30, 16)
Output: direction on day31         → 0 or 1
```

In [ ]:
WINDOW = 30  # lookback window in trading days

def create_sequences(features, labels, window):
    """
    Convert flat arrays into overlapping windows.
    features: (n_days, n_features) array
    labels:   (n_days,) array
    Returns X of shape (n_samples, window, n_features)
            y of shape (n_samples,)
    """
    X, y = [], []
    for i in range(window, len(features)):
        X.append(features[i-window:i])   # past 30 days
        y.append(labels[i])              # tomorrow's direction
    return np.array(X), np.array(y)


# Get target arrays for each split
train_labels = train_df[TARGET_COL].values
val_labels   = val_df[TARGET_COL].values
test_labels  = test_df[TARGET_COL].values

X_train, y_train = create_sequences(train_scaled, train_labels, WINDOW)
X_val,   y_val   = create_sequences(val_scaled,   val_labels,   WINDOW)
X_test,  y_test  = create_sequences(test_scaled,  test_labels,  WINDOW)

print(f'X_train shape: {X_train.shape}  → (samples, window, features)')
print(f'X_val shape:   {X_val.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')

## Step 6 — PyTorch Dataset and DataLoader

In [ ]:
class StockDataset(Dataset):
    """
    PyTorch Dataset wraps our numpy arrays.
    DataLoader uses this to create batches automatically.
    """
    def __init__(self, X, y):
        # Convert numpy → PyTorch tensors
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 32

train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(StockDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(StockDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

# Verify shapes of one batch
sample_X, sample_y = next(iter(train_loader))
print(f'\nSample batch X shape: {sample_X.shape}  → (batch, window, features)')
print(f'Sample batch y shape: {sample_y.shape}  → (batch,)')

## Step 7 — Define the LSTM model

Architecture:
```
Input (30, 16) → LSTM (2 layers, 64 hidden) → Dropout → Dense(32) → Dense(1) → Sigmoid
```
Sigmoid at the end squashes output to [0,1] — perfect for binary classification.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super(LSTMModel, self).__init__()

        # LSTM layer
        # input_size  = number of features per timestep (16)
        # hidden_size = number of memory units in LSTM (64)
        # num_layers  = stacked LSTM layers (2 = deeper model)
        # batch_first = input shape is (batch, seq, features) not (seq, batch, features)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )

        # Dropout to prevent overfitting
        self.dropout = nn.Dropout(dropout)

        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

        # Sigmoid: converts raw score to probability [0, 1]
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch, window, features)

        # LSTM output: all hidden states + final hidden/cell state
        lstm_out, (h_n, c_n) = self.lstm(x)

        # Take only the LAST timestep's output
        # This is the summary of the entire 30-day sequence
        out = lstm_out[:, -1, :]   # shape: (batch, hidden_size)

        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)

        return out.squeeze()


# Instantiate model
INPUT_SIZE = len(FEATURE_COLS)   # 16 features
model = LSTMModel(input_size=INPUT_SIZE).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## Step 8 — Define loss function and optimizer

In [ ]:
# BCELoss = Binary Cross Entropy Loss
# Standard loss for binary classification
criterion = nn.BCELoss()

# Adam optimizer — adaptive learning rate, works well for LSTMs
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler — reduce LR when val loss stops improving
# This helps the model fine-tune in later epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True
)

print('Loss function: BCELoss')
print('Optimizer:     Adam (lr=0.001)')
print('Scheduler:     ReduceLROnPlateau (patience=5)')

## Step 9 — Training loop

In [ ]:
EPOCHS = 50

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10   # stop if no improvement for 10 epochs

for epoch in range(EPOCHS):

    # ── TRAINING PHASE ──
    model.train()
    batch_losses, batch_preds, batch_targets = [], [], []

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()           # clear gradients from last step
        preds = model(X_batch)          # forward pass
        loss  = criterion(preds, y_batch)  # compute loss
        loss.backward()                 # backprop
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()                # update weights

        batch_losses.append(loss.item())
        batch_preds.extend((preds > 0.5).cpu().numpy())
        batch_targets.extend(y_batch.cpu().numpy())

    train_loss = np.mean(batch_losses)
    train_acc  = accuracy_score(batch_targets, batch_preds)

    # ── VALIDATION PHASE ──
    model.eval()
    val_batch_losses, val_preds, val_targets = [], [], []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds   = model(X_batch)
            loss    = criterion(preds, y_batch)
            val_batch_losses.append(loss.item())
            val_preds.extend((preds > 0.5).cpu().numpy())
            val_targets.extend(y_batch.cpu().numpy())

    val_loss = np.mean(val_batch_losses)
    val_acc  = accuracy_score(val_targets, val_preds)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # Step scheduler
    scheduler.step(val_loss)

    # Print every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1:02d}/{EPOCHS}]  '
              f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  '
              f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}')

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_lstm_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

print(f'\nBest val loss: {best_val_loss:.4f}')
print('Best model saved to best_lstm_model.pth')

## Step 10 — Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('LSTM Training Curves', fontsize=13)

epochs_ran = range(1, len(train_losses) + 1)

axes[0].plot(epochs_ran, train_losses, label='Train loss', color='steelblue')
axes[0].plot(epochs_ran, val_losses,   label='Val loss',   color='orange')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, train_accs, label='Train acc', color='steelblue')
axes[1].plot(epochs_ran, val_accs,   label='Val acc',   color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Step 11 — Evaluate on test set

In [ ]:
# Load best saved model
model.load_state_dict(torch.load('best_lstm_model.pth', map_location=device))
model.eval()

all_preds, all_targets, all_probs = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        probs   = model(X_batch).cpu().numpy()
        preds   = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy().astype(int))

# Metrics
acc = accuracy_score(all_targets, all_preds)
f1  = f1_score(all_targets, all_preds)

print('=' * 45)
print('        TEST SET RESULTS (LSTM only)')
print('=' * 45)
print(f'  Accuracy:  {acc:.4f} ({acc*100:.1f}%)')
print(f'  F1 Score:  {f1:.4f}')
print('=' * 45)
print('\nDetailed classification report:')
print(classification_report(all_targets, all_preds, target_names=['DOWN', 'UP']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_targets, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['DOWN', 'UP'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — LSTM on Test Set')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Step 12 — Save everything

In [ ]:
import pickle

# Save scaler (needed for inference later)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save test predictions for Phase 4 analysis
results_df = test_df.iloc[WINDOW:].copy().reset_index(drop=True)
results_df['predicted']   = all_preds
results_df['probability'] = all_probs
results_df['correct']     = (results_df['predicted'] == results_df['target']).astype(int)
results_df.to_csv('lstm_test_predictions.csv', index=False)

print('Saved:')
print('  best_lstm_model.pth       — trained model weights')
print('  scaler.pkl                — fitted MinMaxScaler')
print('  lstm_test_predictions.csv — predictions on test set')
print('  training_curves.png')
print('  confusion_matrix.png')
print(f'\nFinal test accuracy: {acc*100:.1f}%')

## Summary

You now have a trained LSTM that predicts TSLA price direction.

**What to expect accuracy-wise:**
- 50–52% = model is barely better than random (like flipping a coin)
- 53–56% = decent signal, the model is learning something real
- 57%+    = strong result for stock direction prediction

Stock direction prediction is genuinely hard — even hedge funds struggle to consistently beat 55%. Any result above 52% with your dataset size is a good outcome.

**Next — Phase 4:** We fuse FinBERT sentiment + LSTM into one combined model and run the ablation study to show the sentiment features improve accuracy.